# Processing data from the C++ program

In [1]:
using Plots
using Polynomials
using Symbolics
using JLD2
using JuliVirBootstrap

res_dir = "/Users/Paul/Documents/Recherche/projet_these/code/transfer_matrices/TransferMatricesCpp/On_loops/results";
cd(res_dir)

## Checking the central charge

We start by checking the central charge, which we get as

$$ -\frac1L\log \Lambda_0(L) = f_0^\infty - \frac{\pi c}{6L^2} $$

In [ ]:
# load the data, below is for λ = 0.5, n = -2cos(4λ)
Λs = load("ev_wo_defects.jld2", "Λs")
Λ0(L) = Λs[L][1];
println(Λ0(11))

In [ ]:
@variables c1, c2, c3

f0(L) = -log(real(Λ0(L)))/L

# get and effective central charge from sizes L, L+1 and L+2
eq(L) = c1 - π*c2/6/L^2 + c3/L^4 - f0(L)
c_eff(L) = Symbolics.solve_for([eq(L), eq(L+1), eq(L+2)], [c1, c2, c3])[2]

Lrange = 5:10
c_effs = [c_eff(L) for L in Lrange]
Ls = 1 ./ Lrange

# interpolate the central charges to infinite size
deg = length(Lrange)-1
p0 = fit(Ls, c_effs)
println(p0(0))

# Plot to check soundness
xfine = 0:0.001:0.25
scatter(Ls, c_effs)
plot!(xfine, p0.(xfine), xlims=(0, 0.25))

The expected central charge is 

$$
c = 13 - 6(\beta^2 + \beta^{-2})
$$

where $n = -2\cos(\pi \beta^2)$ so $\beta^2 = \frac{4\lambda}\pi$

In [ ]:
λ = 0.5
β = sqrt(4λ/π)
c_exp = 13 - 6*(β^2 + 1/β^2)

The central charge extracted from the transfer matrix matches the expected value to a good accuracy.

## Two-point function of the two-leg operator
### From the spectrum

We extract the dimension of the two-leg operator thanks to

$$
-\frac{1}{L} \log(\frac{\Lambda(L)}{\Lambda_0(L)}) = \frac{2\pi\Delta}{L^2} + o(L^{-2}).
$$

where $\Lambda$ is the highest eigenvalue in the sector with two defects.

In [ ]:
# load the data, below is for λ = 0.5, n = -2cos(4λ)
Λ1s = load("ev_2_defects.jld2", "Λs")
Λ1(L) = Λ1s[L][1];
println(Λ1(6))

In [ ]:
@variables Δ

f1(L) = -log(real(Λ1(L)))/L

# get an effective scaling dimension at size L
eq(L) = f1(L)-f0(L) - π*Δ*2/L^2
Δ_eff(L) = Symbolics.value(Symbolics.solve_for(eq(L), Δ))

Lrange = 5:10
Δ_effs = [Δ_eff(L) for L in Lrange]
Ls = 1 ./ Lrange

# interpolate the central charges to infinite size
deg = length(Lrange)-1
p1 = fit(Ls, Δ_effs)
println(p1(0))

# Plot to check soundness
xfine = 0:0.001:0.25
scatter(Ls, Δ_effs)
plot!(xfine, p1.(xfine), xlims=(0, 0.25))

The expected result is $2\Delta_{(1, 0)} = 0.214601...$. The results match nicely.

In [ ]:
c = CentralCharge(:β, β)
V = Field(c, Kac=true, r=1, s=0)
println(V.Δ[:left] + V.Δ[:right])

### From the two-point function

In [ ]:
# load the data, below is for λ = 0.5, n = -2cos(4λ), M = 300
M = 1000
C10s = load("two_point_2_defects.jld2", "Cs")
C10(L) = C10s[L] 
setprecision(BigFloat, 10, base=10)
logΛ10(L) = log(abs(C10(L)))/M
println(logΛ10(10)) # check that the import went smoothly

In [ ]:
exp(logΛ10(10))/Λ1(10)

In [ ]:
@variables Δ

f10(L) = -logΛ10(L)/L

# get an effective scaling dimension at size L
eq(L) = f10(L)-f0(L) - π*Δ*2/L^2
Δ_eff(L) = Symbolics.value(Symbolics.solve_for(eq(L), Δ))

Lrange = 5:11
Δ_effs = [Δ_eff(L) for L in Lrange]
Ls = 1 ./ Lrange

# interpolate the central charges to infinite size
# deg = length(Lrange)-1
# p1 = fit(Ls, Δ_effs)
# println(p1(0))

# Plot to check soundness
xfine = 0:0.001:0.25
scatter(Ls, Δ_effs, ylims=(0.2, 0.3))
# plot!(xfine, p1.(xfine), xlims=(0, 0.25))

With this method the accuracy is terrible (for now) since I'm doing things in the most naive way possible, but the method can be improved and I trust I can obtain a decent accuracy. The order of magnitude is there though, let's see already if I can get the right order of magnitude for the current.

## Two-point function of the current

I now compute the two-point function $\langle \sigma | T^M | \sigma \rangle$, where $\sigma$ is a normalised state that is odd under the translation by half of the lattice sites.
The goal is to verify whether this two-point function is dominated by the contribution of a state whose dimension converges to the dimension of the current operator $V_{(1, 1)}$.

I expect that
$$
\underbrace{\langle \sigma | T(L)^M | \sigma \rangle}_{=: C(L, M)} = \Lambda^M
$$

where $\Lambda$ corresponds to the dimension of the current.

In [ ]:
# load the data, below is for λ = 0.5, n = -2cos(4λ), M = 300
M = 300
C11s = load("two_point_odd_sector.jld2", "Cs")
C11(L) = C11s[L] 
setprecision(BigFloat, 10, base=10)
logΛ11(L) = log(abs(C11(L)))/M
println(logΛ11(10)) # check that the import went smoothly

In [ ]:
@variables Δ

f11(L) = -logΛ11(L)/L

# get an effective scaling dimension at size L
eq(L) = f11(L)-f0(L) - π*Δ*2/L^2
Δ_eff(L) = Symbolics.value(Symbolics.solve_for(eq(L), Δ))

Lrange = 4:2:10
Δ_effs = [Δ_eff(L) for L in Lrange]
Ls = 1 ./ Lrange

# interpolate the central charges to infinite size
# deg = length(Lrange)-1
# p1 = fit(Ls, Δ_effs)
# println(p1(0))

# Plot to check soundness
xfine = 0:0.001:0.25
scatter(Ls, Δ_effs)
# plot!(xfine, p1.(xfine), xlims=(0, 0.25))

In [ ]:
C11s[4]

In [ ]:
Threads.nthreads()

: 